# 📊 NetlogRAG — Evaluacija i GGUF (v3 modeli)
**Diplomski rad** | Sveučilište Jurja Dobrile u Puli

Ovaj notebook **ne trenira**. Učitava gotov fine-tunani model s HuggingFacea,
evaluira ga i napravi GGUF za Ollamu.

Sve se sprema na **Google Drive**, ne preko `files.download()` — taj poziv
traži aktivnu karticu u pregledniku i zna tiho zakazati.

| Faza | Ćelije | Trajanje |
|---|---|---|
| Priprema + Drive | 1–4 | ~2 min |
| Upload CSV + dataset | 5–7 | ~8 min |
| Učitavanje modela | 8 | ~3 min |
| **Evaluacija** | 9 | ~15 min |
| Izvještaj → Drive | 10 | odmah |
| GGUF → Drive | 11 | ~16 min |

> ⚠️ Runtime → Change runtime type → **T4 GPU**

> 💡 Ako ti treba **samo GGUF**, pokreni ćelije 1–4, pa 8 i 11.
> Ćelije 5–7 i 9–10 su potrebne samo za evaluaciju.


## 1. Instalacija

In [ ]:
%%capture
!pip install unsloth datasets huggingface_hub scikit-learn
print("✅ Gotovo")

## 2. Konfiguracija — odaberi model

Promijeni samo `MODEL_KEY`. Podaci o treningu za sva tri modela su već upisani.

In [ ]:
MODEL_KEY = "phi"          # ← "llama" | "smollm2" | "phi"

HF_USERNAME = "lovro77"
HF_TOKEN    = ""           # ← zalijepi token

MODELS = {
    "llama": {
        "name": "llama32-1b-netlograg-v3",
        "base": "meta-llama/Llama-3.2-1B-Instruct",
        "minutes": 68.9, "steps": 1260,
        "epochs": [(1, 0.131695, 0.140801), (2, 0.117322, 0.128389), (3, 0.116789, 0.125419)],
        "batch": 4, "accum": 4,
    },
    "smollm2": {
        "name": "smollm2-1.7b-netlograg-v3",
        "base": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
        "minutes": 114.1, "steps": 1260,
        "epochs": [(1, 0.129560, 0.126204), (2, 0.108813, 0.111151), (3, 0.101959, 0.107308)],
        "batch": 4, "accum": 4,
    },
    "phi": {
        "name": "phi35-mini-netlograg-v3",
        "base": "microsoft/Phi-3.5-mini-instruct",
        "minutes": 236.2, "steps": 1260,
        "epochs": [(1, 0.110806, 0.106884), (2, 0.092316, 0.095044), (3, 0.086499, 0.091994)],
        "batch": 2, "accum": 8,
    },
}

CFG        = MODELS[MODEL_KEY]
MODEL_NAME = CFG["name"]
MODEL_ID   = CFG["base"]
EPOCHS_LOG = [{"epoch": float(e), "train_loss": t, "eval_loss": v} for e, t, v in CFG["epochs"]]
TRAIN_MINUTES, TOTAL_STEPS, TRAIN_EXAMPLES = CFG["minutes"], CFG["steps"], 6720
BATCH_SIZE, GRAD_ACCUM = CFG["batch"], CFG["accum"]

# Moraju odgovarati treningu v3
MAX_SEQ_LENGTH   = 512
SAMPLE_PER_CLASS = 1200
INCLUDE_PORT     = False
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE, NUM_EPOCHS, WARMUP_RATIO = 2e-4, 3, 0.1

EVAL_N = 60      # 60 je dovoljno; 200 traje predugo

print(f"Model:  {HF_USERNAME}/{MODEL_NAME}")
print(f"Bazni:  {MODEL_ID}")
print(f"Trening: {TRAIN_MINUTES} min, zadnji eval loss {EPOCHS_LOG[-1]['eval_loss']}")

## 3. HuggingFace login

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("✅ Prijavljen")
else:
    print("⚠️ Upisi HF_TOKEN u celiju 2!")

## 4. Google Drive

Sve izlazne datoteke idu ovdje. Za razliku od `files.download()`, ovo radi
i kad je kartica u pozadini ili kad sesija traje satima.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/NetlogRAG"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"✅ Izlazi: {OUT_DIR}")

## 5. Upload CSV-ova

**Istih pet fajlova kao pri treningu v3**: Tuesday, Wednesday,
Thursday-WebAttacks, Friday-PortScan, Friday-DDos.

*(Preskoci ako radis samo GGUF.)*

In [ ]:
from google.colab import files
import pandas as pd

KEEP_COLS = ["Label","Protocol","Destination Port","Flow Duration",
 "Total Fwd Packets","Total Backward Packets","Total Length of Fwd Packets",
 "Total Length of Bwd Packets","Fwd Packet Length Mean","Bwd Packet Length Mean",
 "Flow Bytes/s","Flow Packets/s","Flow IAT Mean","Flow IAT Std",
 "SYN Flag Count","PSH Flag Count","ACK Flag Count","Down/Up Ratio",
 "Average Packet Size"]

def norm_label(s):
    s = str(s).strip(); low = s.lower()
    if "web attack" in low:
        if "brute" in low: return "Web Attack - Brute Force"
        if "xss"   in low: return "Web Attack - XSS"
        if "sql"   in low: return "Web Attack - Sql Injection"
    return s

uploaded = files.upload()

dfs = []
for filename in sorted(uploaded):
    df = pd.read_csv(filename, low_memory=False, encoding="latin-1")
    df = df.rename(columns={c: c.strip() for c in df.columns})
    df = df.loc[:, ~df.columns.duplicated()]
    df = df[[c for c in KEEP_COLS if c in df.columns]]
    print(f"✅ {filename}: {len(df):,} redova")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True); del dfs
df_all["Label"] = df_all["Label"].apply(norm_label)
print(f"\nUKUPNO: {len(df_all):,} redova, {df_all['Label'].nunique()} labela")

## 6. Rekonstrukcija dataseta

Identicno treningu v3 — isti `random_state=42`, isti `seed(42)`, port iskljucen.
Zato ispadne isti test set.

In [ ]:
import json, random
import numpy as np
from collections import Counter

FLOW_FEATURES = ["Flow Duration","Total Fwd Packets","Total Backward Packets",
 "Total Length of Fwd Packets","Total Length of Bwd Packets","Fwd Packet Length Mean",
 "Bwd Packet Length Mean","Flow Bytes/s","Flow Packets/s","Flow IAT Mean",
 "Flow IAT Std","SYN Flag Count","PSH Flag Count","ACK Flag Count",
 "Down/Up Ratio","Average Packet Size"]

ATTACK_MAP = {
 "BENIGN":("BENIGN","LOW","Normalni mrezni promet bez znakova prijetnje."),
 "FTP-Patator":("FTP-Patator","HIGH","Brute force napad na FTP servis."),
 "SSH-Patator":("SSH-Patator","HIGH","Brute force napad na SSH servis."),
 "DoS Hulk":("DoS","HIGH","DoS Hulk napad."),
 "DoS GoldenEye":("DoS","HIGH","DoS GoldenEye napad."),
 "DoS slowloris":("DoS","HIGH","Slowloris napad."),
 "DoS Slowhttptest":("DoS","HIGH","Slow HTTP DoS napad."),
 "Heartbleed":("DoS","HIGH","Heartbleed."),
 "DDoS":("DDoS","HIGH","Distribuirani DoS napad."),
 "PortScan":("PortScan","MEDIUM","Skeniranje portova."),
 "Bot":("Botnet","HIGH","Botnet aktivnost."),
 "Infiltration":("Infiltration","HIGH","Infiltracija mreze."),
 "Web Attack - Brute Force":("WebAttack","HIGH","Brute force na web autentikaciju."),
 "Web Attack - XSS":("WebAttack","HIGH","Cross-site scripting napad."),
 "Web Attack - Sql Injection":("WebAttack","HIGH","SQL Injection napad.")}

INDICATORS = {"BENIGN":["Uobicajeno trajanje toka","Uravnotezen omjer paketa"],
 "FTP-Patator":["Ponavljajuce kratke veze","Visok broj pokusaja autentikacije"],
 "SSH-Patator":["Ponavljajuce kratke veze","Uzastopni pokusaji prijave"],
 "DoS":["Visok broj zahtjeva po sekundi","Kratki tokovi velike ucestalosti"],
 "DDoS":["Vrlo visok SYN broj","Kratko trajanje uz veliku propusnost"],
 "PortScan":["Minimalan prijenos podataka","Vrlo kratki tokovi"],
 "Botnet":["Periodicni obrazac prometa","Pravilni vremenski intervali"],
 "WebAttack":["Neuobicajena velicina paketa","Anomalan HTTP obrazac"],
 "Infiltration":["Neuobicajen odlazni promet","Netipicno trajanje sesije"]}

ACTIONS = {"BENIGN":["Nastaviti redovni monitoring"],
 "FTP-Patator":["Blokirati izvornu IP adresu","Uvesti rate limiting na FTP"],
 "SSH-Patator":["Blokirati izvornu IP adresu","Prijeci na autentikaciju kljucem"],
 "DoS":["Aktivirati WAF","Uvesti rate limiting"],
 "DDoS":["Aktivirati DDoS zastitu","Kontaktirati ISP"],
 "PortScan":["Blokirati izvornu IP adresu","Pregledati firewall pravila"],
 "Botnet":["Izolirati zarazeni uredaj","Blokirati C&C domenu"],
 "WebAttack":["Aktivirati WAF","Pregledati validaciju ulaza"],
 "Infiltration":["Izolirati pogodeni segment","Pokrenuti forenzicku analizu"]}

PROTO_MAP = {6:"TCP",17:"UDP",1:"ICMP",0:"HOPOPT"}

def safe_float(v):
    if v is None: return None
    try: x = float(v)
    except (TypeError, ValueError): return None
    return None if (np.isnan(x) or np.isinf(x)) else x

def build_instruction(row):
    label = str(row.get("Label","BENIGN")).strip()
    attack, risk, summary = ATTACK_MAP.get(label, ("Unknown","MEDIUM",f"Detektiran {label}."))
    parts = []
    if INCLUDE_PORT:
        p = safe_float(row.get("Destination Port"))
        parts.append(f"port={int(p) if p is not None else 0}")
    for f in FLOW_FEATURES:
        v = safe_float(row.get(f))
        if v is None: continue
        s = (f.replace("Total ","").replace("Length of ","Len").replace("Packet","Pkt")
              .replace("Backward","Bwd").replace("Forward","Fwd").replace(" ",""))
        parts.append(f"{s}={v:.1f}")
    pv = safe_float(row.get("Protocol"))
    proto = PROTO_MAP.get(int(pv),"OTHER") if pv is not None else "TCP"
    input_text = f"{proto} tok. " + ", ".join(parts)
    output = json.dumps({"attack_type":attack,"risk_level":risk,"summary":summary,
        "key_indicators":INDICATORS.get(attack,["Anomalan obrazac toka"]),
        "recommended_actions":ACTIONS.get(attack,["Istraziti aktivnost"])},
        ensure_ascii=False)
    text = ("### Instruction:\nAnaliziraj karakteristike ovog mreznog toka, "
            "odredi tip napada i procijeni razinu rizika. Vrati ISKLJUCIVO JSON.\n\n"
            f"### Input:\n{input_text}\n\n### Response:\n{output}")
    return {"text":text,"attack_type":attack,"risk_level":risk}

df_all["_attack"] = df_all["Label"].map(lambda l: ATTACK_MAP.get(str(l).strip(),("Unknown",))[0])

unknown = df_all[df_all["_attack"]=="Unknown"]["Label"].value_counts()
if len(unknown):
    print("⚠️ NEPREPOZNATE LABELE:")
    for lbl, c in unknown.items(): print(f"   {repr(lbl)} → {c:,}")
else:
    print("✅ Sve labele prepoznate")

records, errors = [], []
for attack, group in df_all.groupby("_attack"):
    if attack == "Unknown": continue
    n = min(len(group), SAMPLE_PER_CLASS)
    for _, row in group.sample(n=n, random_state=42).iterrows():
        try: records.append(build_instruction(row))
        except Exception as e: errors.append(f"{attack}: {e}")

random.seed(42); random.shuffle(records)
if errors: print(f"⚠️ {len(errors)} preskoceno")
print(f"\n✅ {len(records):,} primjera")
for k, v in Counter(r["attack_type"] for r in records).most_common():
    print(f"  {k:<14}{v:>5}")
assert "port=" not in records[0]["text"], "PORT CURI!"

## 7. Test set (isti split kao pri treningu)

In [ ]:
from datasets import Dataset
from collections import defaultdict

by_class = defaultdict(list)
for r in records: by_class[r["attack_type"]].append(r)

train_recs, val_recs, test_recs = [], [], []
for attack, items in by_class.items():
    random.seed(42); random.shuffle(items)
    n_tr = int(len(items)*0.80); n_va = int(len(items)*0.10)
    train_recs += items[:n_tr]
    val_recs   += items[n_tr:n_tr+n_va]
    test_recs  += items[n_tr+n_va:]
for l in (train_recs, val_recs, test_recs): random.shuffle(l)

train_dataset = Dataset.from_list(train_recs)
val_dataset   = Dataset.from_list(val_recs)
test_dataset  = Dataset.from_list(test_recs)
print(f"Train {len(train_dataset):,} | Val {len(val_dataset):,} | Test {len(test_dataset):,}")
print(dict(Counter(test_dataset["attack_type"])))

## 8. Ucitavanje fine-tunanog modela s Huba

Ovdje se ucitava **fine-tunani** model, ne bazni — LoRA adapteri su u njemu.

In [ ]:
from unsloth import FastLanguageModel
import torch, time

REPO = f"{HF_USERNAME}/{MODEL_NAME}"
print(f"Ucitavam: {REPO}")

t0 = time.perf_counter()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
LOAD_SECONDS = round(time.perf_counter()-t0, 1)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Ucitan za {LOAD_SECONDS}s | VRAM {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 9. Evaluacija

Racuna tocnost tipa napada i razine rizika, macro F1, precision i recall
po klasi te latenciju s warm-upom i `cuda.synchronize()`.

`extract_json` reze izlaz na prvom potpunom JSON objektu jer model
nastavi generirati i nakon njega.

In [ ]:
import json, time, torch
from tqdm.auto import tqdm
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support

FastLanguageModel.for_inference(model)

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        p = text.split("```")
        if len(p) > 1:
            text = p[1][4:] if p[1].startswith("json") else p[1]
            text = text.strip()
    depth = 0; start = None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                return text[start:i+1]
    return None

# warm-up
_w = tokenizer("### Instruction:\ntest\n\n### Response:\n", return_tensors="pt").to("cuda")
with torch.no_grad():
    model.generate(**_w, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()

n_eval = min(EVAL_N, len(test_dataset))
y_true, y_pred, lat = [], [], []
risk_ok = parsed = 0

for item in tqdm(test_dataset.select(range(n_eval)), desc="Evaluiram"):
    prompt = item["text"].split("### Response:")[0] + "### Response:\n"
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=MAX_SEQ_LENGTH).to("cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=220, temperature=0.1,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize(); lat.append((time.perf_counter()-t0)*1000)

    gen = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    raw = extract_json(gen)
    pred = "PARSE_FAIL"
    if raw:
        try:
            d = json.loads(raw); parsed += 1
            pred = d.get("attack_type", "PARSE_FAIL")
            if d.get("risk_level") == item["risk_level"]: risk_ok += 1
        except json.JSONDecodeError:
            pass
    y_true.append(item["attack_type"]); y_pred.append(pred)

labels = sorted(set(y_true))
p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
_, _, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels,
                                                    average="macro", zero_division=0)
acc = sum(a == b for a, b in zip(y_true, y_pred)) / n_eval * 100
ls = sorted(lat)

conf = defaultdict(int)
for t, pr in zip(y_true, y_pred): conf[f"{t}->{pr}"] += 1

EVAL = {
    "attack_type_accuracy_pct": round(acc, 1),
    "risk_level_accuracy_pct":  round(risk_ok/n_eval*100, 1),
    "macro_f1":                 round(float(macro_f1), 4),
    "json_parse_pct":           round(parsed/n_eval*100, 1),
    "n_samples":                n_eval,
    "latency_mean_ms":          round(sum(ls)/len(ls), 1),
    "latency_p50_ms":           round(ls[len(ls)//2], 1),
    "latency_p95_ms":           round(ls[int(len(ls)*0.95)], 1),
    "per_class": {c: {"precision": round(float(p[i]),4), "recall": round(float(r[i]),4),
                      "f1": round(float(f1[i]),4), "support": int(sup[i])}
                  for i, c in enumerate(labels)},
    "confusion": dict(sorted(conf.items(), key=lambda x: -x[1])),
}

print(f"\n{'='*58}")
print(f"  Tip napada:    {EVAL['attack_type_accuracy_pct']}%")
print(f"  Razina rizika: {EVAL['risk_level_accuracy_pct']}%")
print(f"  Macro F1:      {EVAL['macro_f1']}")
print(f"  JSON parse:    {EVAL['json_parse_pct']}%")
print(f"  Latencija:     {EVAL['latency_mean_ms']} ms (p95 {EVAL['latency_p95_ms']})")
print(f"{'='*58}")
print(f"{'Klasa':<14}{'Prec':>8}{'Rec':>8}{'F1':>8}{'N':>5}")
print("-"*45)
for c, m in sorted(EVAL["per_class"].items(), key=lambda x: -x[1]["support"]):
    print(f"{c:<14}{m['precision']:>8.3f}{m['recall']:>8.3f}{m['f1']:>8.3f}{m['support']:>5}")

errs = [(k, v) for k, v in EVAL["confusion"].items() if k.split("->")[0] != k.split("->")[1]]
if errs:
    print("\nGRESKE:")
    for k, v in errs:
        t, pr = k.split("->")
        kind = "propusten napad" if pr == "BENIGN" else ("lazni alarm" if t == "BENIGN" else "zamjena klase")
        print(f"  {t:<14} → {pr:<14} {v:>3}x   ({kind})")

## 10. Izvjestaj → Google Drive

In [ ]:
import json, torch, shutil
from datetime import datetime
from collections import Counter

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

report = {
    "version":      "v3 — bez porta, 7 klasa (s WebAttack), predikcija tipa napada",
    "timestamp":    datetime.now().isoformat(),
    "model_id":     MODEL_ID,
    "model_name":   MODEL_NAME,
    "hf_repo":      f"{HF_USERNAME}/{MODEL_NAME}",
    "gpu":          torch.cuda.get_device_name(0),
    "vram_peak_gb": round(torch.cuda.max_memory_allocated()/1024**3, 2),
    "load_time_s":  LOAD_SECONDS,
    "hyperparameters": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "max_seq_length": MAX_SEQ_LENGTH, "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM, "effective_batch": BATCH_SIZE*GRAD_ACCUM,
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
    },
    "model_params": {"total": total, "trainable": trainable,
                     "trainable_pct": round(trainable/total*100, 3)},
    "dataset": {
        "source": "CICIDS2017 (5 dana)", "include_port": INCLUDE_PORT,
        "n_flow_features": len(FLOW_FEATURES), "sample_per_class": SAMPLE_PER_CLASS,
        "total_examples": len(records), "train": len(train_dataset),
        "validation": len(val_dataset), "test": len(test_dataset),
        "attack_distribution": dict(Counter(r["attack_type"] for r in records)),
        "risk_distribution":   dict(Counter(r["risk_level"]  for r in records)),
    },
    "training": {"minutes": TRAIN_MINUTES, "total_steps": TOTAL_STEPS,
                 "train_examples": TRAIN_EXAMPLES, "epochs_log": EPOCHS_LOG},
    "evaluation": EVAL,
}

fname = f"report_{MODEL_NAME}.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
shutil.copy(fname, f"{OUT_DIR}/{fname}")
print(f"✅ Na Drive: {OUT_DIR}/{fname}")

## 11. GGUF → Google Drive

⏱️ ~16 minuta. Ide na Drive, ne preko preglednika.

In [ ]:
import glob, os, shutil

model.save_pretrained_gguf(f"{MODEL_NAME}-gguf", tokenizer,
                           quantization_method="q4_k_m")

found = [f for f in glob.glob(f"{MODEL_NAME}-gguf*/**/*.gguf", recursive=True)
         if "Q4_K_M" in f or "q4_k_m" in f]

if found:
    dest = f"{OUT_DIR}/{MODEL_NAME}-Q4_K_M.gguf"
    shutil.copy(found[0], dest)
    print(f"✅ Na Drive: {dest} ({os.path.getsize(dest)/1024**3:.2f} GB)")
else:
    print("⚠️ GGUF nije pronaden:")
    for f in glob.glob("**/*.gguf", recursive=True):
        print("  ", f, f"({os.path.getsize(f)/1024**3:.2f} GB)")

## 12. Provjera i gasenje sesije

In [ ]:
import os, time

print(f"Sadrzaj {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f}  ({os.path.getsize(f'{OUT_DIR}/{f}')/1024**2:.1f} MB)")

print("\nGasim sesiju za 60 s...")
time.sleep(60)
from google.colab import runtime
runtime.unassign()

## 13. Sljedeci model

Vrati se na **celiju 2**, promijeni `MODEL_KEY` u `"llama"` ili `"smollm2"`,
pa Runtime → Restart session i pokreni ponovo.

Restart je bitan — bez njega VRAM od prethodnog modela ostaje zauzet,
sto je vjerojatno i srusilo evaluaciju Phi-ja odmah nakon treninga.